In [ ]:
from napari.utils.colormaps import Colormap
import numpy as np
import xarray as xr

import napari
import numpy as np

viewer = napari.Viewer()


napari.run()




In [ ]:

def box_edges(min_corner, max_corner):
    z0, y0, x0 = min_corner
    z1, y1, x1 = max_corner

    corners = np.array([
        [z0, y0, x0],  # 0
        [z0, y0, x1],  # 1
        [z0, y1, x1],  # 2
        [z0, y1, x0],  # 3
        [z1, y0, x0],  # 4
        [z1, y0, x1],  # 5
        [z1, y1, x1],  # 6
        [z1, y1, x0],  # 7
    ], dtype=float)

    edge_ids = [
        (0, 1), (1, 2), (2, 3), (3, 0),  # front face
        (4, 5), (5, 6), (6, 7), (7, 4),  # back face
        (0, 4), (1, 5), (2, 6), (3, 7),  # connecting edges
    ]

    return [corners[[i, j]] for i, j in edge_ids]
vertices = np.array([
    [0, 0, 0], [0, 0, 10], [0, 10, 0], [0, 10, 10],
    [10, 0, 0], [10, 0, 10], [10, 10, 0], [10, 10, 10]
])
size = 256
x,y,z = 0,0,0
lines = box_edges(min_corner=(x, y, z), max_corner=(x+size, y+size, z+size))
viewer.add_shapes(
    box,
    shape_type="path",
    edge_color="grey",
    face_color="transparent",
    edge_width=1,
)

In [19]:
colors_bop_gold = np.linspace(
    start=[1, 0.84, 0, 0.5],
    stop=[1, 0.84, 0, 1],
    num=10,
    endpoint=True
)
colors_gold = np.linspace(
    start=[1, 1, 1, 1],
    stop=[1, 0.84, 0, 1],
    num=10,
    endpoint=True
)
viewer.layers[2].colormap = colors_bop_gold

In [ ]:
save_folder = r'C:\Users\TCraig\Pictures\Visualizations'
viewer.screenshot(save_folder + r'\napari_visualization.png')

viewer.camera.perspective = 0


In [ ]:
save_folder = r'C:\Users\TCraig\Pictures\Visualizations'
sample_folder = 'NS_ortho'

n = viewer.layers[0].data.shape[0]

for i in range(n):
    step = list(viewer.dims.current_step)
    step[0] = i

    viewer.dims.current_step = step
    viewer.screenshot(save_folder + rf'\{sample_folder}\orthoslice_time{i}slice{step[1]}.png')



In [ ]:
save_folder = r'C:\Users\TCraig\Pictures\Visualizations'
sample_folder = 'NS_volume'

n = viewer.layers[0].data.shape[0]

viewer.camera.perspective = 30
for i in range(n):
    step = list(viewer.dims.current_step)
    step[0] = i

    viewer.dims.current_step = step
    viewer.screenshot(save_folder + rf'\{sample_folder}\volume_time{i}.png')





In [ ]:
from tomondt.data import VolumeTimeSeries

r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D.vmf'
vnd = VolumeTimeSeries.read(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf')


def center_crop_3d(da, half_width=50):
    crop_slices = {}
    
    for dim in ["z", "y", "x"]:
        n = da.sizes[dim]
        c = n // 2
        start = max(0, c - half_width)
        stop = min(n, c + half_width)
        crop_slices[dim] = slice(start, stop)
    
    return da.isel(**crop_slices)


data = vnd.data.transpose("indices", "x", "y", "z")
data = center_crop_3d(data, half_width=50)

path = r'D:\Assets\Cage-D_processed.zarr'
name = 'Cage-D_sage_forest_34_processed'

for i in range(data.sizes['indices']):
    if i == 0:
        subset = data.isel(indices=i).compute().values[None, ...]
        vndt_2 = VolumeTimeSeries(path=path, name='processed_cage', data=subset)
    else:
        subset = data.isel(indices=[i]).compute()
        subset.to_zarr(vndt_2.path, append_dim='indices', mode='a')


print(vndt_2.data.shape)





2026-03-17 16:17:57,581 - DEBUG - Attempting to load file \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with {'.vmf': <function _read_vmf at 0x0000018BDAF69800>, '.zarr': <function _read_zarr at 0x0000018BE08F8040>, '.nc': <function _read_netcdf at 0x0000018BE08F8900>}
2026-03-17 16:17:59,396 - DEBUG - Reading VMF from \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with name DIP128 and times 100
2026-03-17 16:17:59,396 - DEBUG - Scheduled read of time 1.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 16:17:59,396 - DEBUG - Scheduled read of time 2.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 16:17:59,403 - DEBUG - Scheduled read of time 3.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 16:17:59,403 - DEBUG - Scheduled read of time 4.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 16:17:59,403 - DEBUG - 

(1, 100, 100, 100)


2026-03-17 16:17:59,686 - DEBUG - Set context to GPUContext.NUMPY on device 0
c:\Users\TCraig\AppData\Local\miniconda3\envs\dev-tdtomo2\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-03-17 16:17:59,755 - DEBUG - Attempting to load file D:\Assets\Cage-D_processed.zarr with {'.vmf': <function _read_vmf at 0x0000018BDAF69800>, '.zarr': <function _read_zarr at 0x0000018BE08F8040>, '.nc': <function _read_netcdf at 0x0000018BE08F8900>}
2026-03-17 16:17:59,772 - DEBUG - Set context to GPUContext.NUMPY on device 0


('indices', 'x', 'y', 'z') Frozen({'indices': 1, 'x': 100, 'y': 100, 'z': 100})


c:\Users\TCraig\AppData\Local\miniconda3\envs\dev-tdtomo2\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


ValueError: conflicting sizes for dimension 'indices': length 2 on 'indices' and length 1 on {'indices': 'array-91ae2e4b33c3bdb18a1d59394848b98b', 'z': 'array-91ae2e4b33c3bdb18a1d59394848b98b', 'y': 'array-91ae2e4b33c3bdb18a1d59394848b98b', 'x': 'array-91ae2e4b33c3bdb18a1d59394848b98b'}